# Random Survival Forest (RSF) Training and Evaluation

This notebook implements RSF on the AIDS dataset using the same training/test/validation split and preprocessing as the DeepCoxPH baseline for a fair comparison.

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv
from sksurv.metrics import (
    concordance_index_censored, 
    cumulative_dynamic_auc, 
    integrated_brier_score, 
    brier_score,
    concordance_index_ipcw,
)
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer

# Settings
plt.style.use('seaborn-v0_8-whitegrid')
warnings.filterwarnings("ignore")

In [9]:
# 1. Load Data
df = pd.read_csv("AIDS_Classification_50000.csv")
time_col = 'time'
event_col = 'infected'

X = df.drop(columns=[time_col, event_col])
t = df[time_col].values
e = df[event_col].values

cat_cols = [col for col in X.columns if X[col].nunique() < 10]
num_cols = [col for col in X.columns if col not in cat_cols]

# 2. Split Data (Standardized 70/15/15)
x_train_raw, x_temp, t_train, t_temp, e_train, e_temp = train_test_split(
    X, t, e, test_size=0.30, random_state=42, stratify=e
)
x_test_raw, x_val_raw, t_test, t_val, e_test, e_val = train_test_split(
    x_temp, t_temp, e_temp, test_size=0.30, random_state=42, stratify=e_temp
)

# Remove max values from test/val (consistency with DeepCph baseline)
mask_test = t_test < t_train.max()
x_test_raw = x_test_raw[mask_test]
t_test = t_test[mask_test]
e_test = e_test[mask_test]

mask_val = t_val < t_train.max()
x_val_raw = x_val_raw[mask_val]
t_val = t_val[mask_val]
e_val = e_val[mask_val]

# 3. Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols),
    ]
)

x_train = preprocessor.fit_transform(x_train_raw)
x_test = preprocessor.transform(x_test_raw)
x_val = preprocessor.transform(x_val_raw)

# Prepare sksurv targets
y_train_surv = Surv.from_arrays(e_train.astype(bool), t_train)
y_test_surv = Surv.from_arrays(e_test.astype(bool), t_test)
y_val_surv = Surv.from_arrays(e_val.astype(bool), t_val)

In [ ]:
# # 4. Hyperparameter Tuning using GridSearchCV
# print("Starting Hyperparameter Tuning for RSF...")

# rsf = RandomSurvivalForest(random_state=42, n_jobs=-1)

# param_grid = {
#     "n_estimators": [100, 200, 300],
#     "max_depth": [5, 10, None],
#     "min_samples_split": [10, 20],
#     "min_samples_leaf": [10, 15],
#     "max_features": ["sqrt", "log2"]
# }

# # Explicitly define the scorer using Harrell's C-index
# # concordance_index_censored expects (event, time, risk_score)
# # RandomSurvivalForest.predict returns risk scores by default
# def score_survival_model(model, X, y):
#     prediction = model.predict(X)
#     event = []
#     time = []
#     for i in range(len(y)):
#         event.append(y[i][0])
#         time.append(y[i][1])
#     result = concordance_index_censored(event, time, prediction)
#     return result[0]

# gcv = GridSearchCV(
#     rsf, 
#     param_grid, 
#     cv=3, 
#     scoring=score_survival_model,
#     # n_jobs=-1,
#     verbose=3
# )

# gcv.fit(x_train, y_train_surv)

# print("Tuning Complete.")
# print(f"Best Parameters: {gcv.best_params_}")
# print(f"Best CV Score (C-index): {gcv.best_score_:.4f}")

# # Display top 10 results in a dataframe
# results_df = pd.DataFrame(gcv.cv_results_)
# display(results_df.sort_values(by="rank_test_score").head(10))

# best_rsf = gcv.best_estimator_

Best Parameters: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 10, 'min_samples_split': 10, 'n_estimators': 300}

In [ ]:
# 4. Model Training
best_rsf = RandomSurvivalForest(
    max_depth=10,
    max_features="sqrt",
    min_samples_leaf=10,
    min_samples_split=10,
    n_estimators=300,
    random_state=42
)
best_rsf.fit(x_train, y_train_surv)

RandomSurvivalForest(max_depth=10, min_samples_leaf=10, min_samples_split=10,
                     n_estimators=300, random_state=42)

In [ ]:
# 5. Evaluation of Best Model at Quartiles
horizons = [0.25, 0.50, 0.75]
times = np.quantile(t[e == 1], horizons).tolist()
print(f"Evaluation Horizons: {times}")

# Survival Functions and Risk Scores
risk_scores = best_rsf.predict(x_test)
surv_funcs = best_rsf.predict_survival_function(x_test)

# Get probability of survival at specific 'times'
surv_probs = np.array([[fn(t) for t in times] for fn in surv_funcs])
risk_at_times = 1 - surv_probs # Cumulative incidence approximation for AUC

# For global C-index, use the single risk score
global_risk = risk_scores

# Initialize lists for results
results_quartile = []
results_global = []

# --- Quartile Metrics ---
for i, t_val in enumerate(times):

        current_risk = risk_at_times[:, i]

        # Quartile C-index
        ctd = concordance_index_ipcw(y_train_surv, y_test_surv, current_risk, t_val)[0]

        # Quartile Brier Score
        bs = brier_score(y_train_surv, y_test_surv, surv_probs[:, i].reshape(-1, 1), [t_val])[1][0]

        # Quartile AUC
        auc_val_tuple = cumulative_dynamic_auc(y_train_surv, y_test_surv, current_risk, t_val)
        auc = auc_val_tuple[0][0]

        # Append results
        results_quartile.append({
            "Quartile_Time": t_val,
            "Quartile_Idx": f"Q{i+1}",
            "C-Index (TD)": ctd,
            "Brier Score": bs,
            "AUC Score": auc,
        })

# --- Global Metrics ---
# Integrated Brier / AUC
ibs = integrated_brier_score(y_train_surv, y_test_surv, surv_probs, times)

# Integrated AUC
_, i_auc = cumulative_dynamic_auc(y_train_surv, y_test_surv, risk_at_times, times)

# Global Concordance (Traditional C-index for censored data)
# Note: For time-varying risk models, this is an approximation using a representative risk score
c_global = concordance_index_censored(y_test_surv["event"], y_test_surv["time"], global_risk)[0]

# Append global results
results_global.append({
    "Integrated Brier Score": ibs,
    "Integrated ROC-AUC": i_auc,
    "Global C-Index": c_global
})

df_quartile = pd.DataFrame(results_quartile)
df_global = pd.DataFrame(results_global)

Evaluation Horizons: [496.0, 992.0, 1127.0]


In [19]:
print("--- Metrics by Quartile ---")
display(df_quartile)

print("\n--- Global Metrics (Without Quartiles) ---")
display(df_global)

--- Metrics by Quartile ---


,Quartile_Time,Quartile_Idx,C-Index (TD),Brier Score,AUC Score
0,496.0,Q1,0.709207,0.070633,0.717277
1,992.0,Q2,0.691415,0.139229,0.703476
2,1127.0,Q3,0.659848,0.205476,0.658145



--- Global Metrics (Without Quartiles) ---


,Integrated Brier Score,Integrated ROC-AUC,Global C-Index
0,0.119355,0.687279,0.673661
